In [1]:
import os
#  Get the data 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

/kaggle/input/notebooks/ghalaalbishri/ghala-vc/stress1_model.h5
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/__results__.html
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/stress1.weights.h5
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/__notebook__.ipynb
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/stress1_model.keras
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/__output__.json
/kaggle/input/notebooks/ghalaalbishri/ghala-vc/custom.css
/kaggle/input/datasets/orvile/ravdess-dataset/Video_Speech_Actor_15/Actor_15/02-01-08-02-02-01-15.mp4
/kaggle/input/datasets/orvile/ravdess-dataset/Video_Speech_Actor_15/Actor_15/02-01-05-01-01-02-15.mp4
/kaggle/input/datasets/orvile/ravdess-dataset/Video_Speech_Actor_15/Actor_15/02-01-03-02-02-01-15.mp4
/kaggle/input/datasets/orvile/ravdess-dataset/Video_Speech_Actor_15/Actor_15/02-01-08-01-01-01-15.mp4
/kaggle/input/datasets/orvile/ravdess-dataset/Video_Speech_Actor_15/Actor_15/02-01-07-01-01-02-15.mp4
/kaggle/input/datasets/orvile/ravdess-data

In [2]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

# Define the base path for the RAVDESS dataset
data_path = '/kaggle/input/datasets/orvile/ravdess-dataset'

# Initialize lists to store file locations and their corresponding labels
filepaths = []
labels = []

# Function to extract the emotion label from the filename based on RAVDESS naming conventions
def get_label(filename):
    parts = filename.split('-')
    emotion = int(parts[2]) # The 3rd part of the filename represents the emotion ID

    # Group specific emotions into a binary classification: Stressed vs. Not Stressed
    if emotion in [4, 5, 6]:   # 4=Sad, 5=Angry, 6=Fearful
        return 1               # Classified as 'Stressed'
    elif emotion in [2, 3]:    # 2=Calm, 3=Happy
        return 0               # Classified as 'Not Stressed'
    else:
        return None            # Ignore other emotions (like Neutral or Surprised)

# Walk through the dataset directory to find video files
for root, dirs, files in os.walk(data_path):
    # Only process files within the 'Video_Speech' folders
    if 'Video_Speech' not in root:
        continue

    for file in files:
        # Filter for MP4 video files only
        if file.endswith('.mp4'):
            label = get_label(file)
            
            # If the emotion is one we are tracking, save the path and label
            if label is None:
                continue

            filepaths.append(os.path.join(root, file))
            labels.append(label)

# Split the data: 80% for training the model and 20% for testing its performance
train_p, test_p, train_l, test_l = train_test_split(filepaths, labels, test_size=0.2, random_state=42)

# Display the total count of valid video samples found
print(f"Total Videos: {len(filepaths)}")


Total Videos: 1920


In [3]:
import cv2
import tensorflow as tf
from tensorflow.keras.utils import Sequence
# Custom Data Generator class to handle video loading and preprocessing efficiently
class VideoFeatureGenerator(Sequence):
    def __init__(self, paths, labels, batch_size=4, n_frames=15):
        self.paths = paths           # List of full paths to the video files
        self.labels = labels         # Corresponding stress labels (0 or 1)
        self.batch_size = batch_size # Number of videos to process in each step
        self.n_frames = n_frames     # Number of frames to extract from each video

    # Returns the total number of batches per epoch
    def __len__(self):
        return len(self.paths) // self.batch_size

    # Logic to generate one batch of data
    def __getitem__(self, idx):
        # Select the specific paths and labels for the current batch
        batch_paths = self.paths[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_labels = self.labels[idx*self.batch_size:(idx+1)*self.batch_size]
        
        X = [] # List to store processed video frames
        for path in batch_paths:
            cap = cv2.VideoCapture(path) # Open the video file
            frames = []
            
            # Extract frames until the target 'n_frames' is reached
            while len(frames) < self.n_frames:
                ret, frame = cap.read()
                if not ret: 
                    break # Stop if the video ends early
                
                # Resize frame to 128x128 and normalize pixel values (0 to 1)
                frame = cv2.resize(frame, (128, 128))
                frames.append(frame / 255.0)
            
            cap.release() # Close the video file to free up memory

            # Padding: If the video is too short, fill the remaining frames with black (zeros)
            while len(frames) < self.n_frames:
                frames.append(np.zeros((128, 128, 3)))
            
            X.append(frames)
        
        # Return the batch as NumPy arrays (ready for model training)
        return np.array(X), np.array(batch_labels)

# Initialize the generators for training and testing sets
train_gen = VideoFeatureGenerator(train_p, train_l)
test_gen = VideoFeatureGenerator(test_p, test_l)

2026-05-11 16:48:55.310720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778518135.542402      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778518135.605907      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778518136.124223      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778518136.124271      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778518136.124274      57 computation_placer.cc:177] computation placer alr

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, GlobalAveragePooling2D, GRU, Dense, Dropout, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Build the Model Architecture
# Load MobileNetV2 as a base feature extractor with pre-trained ImageNet weights
base_cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_cnn.trainable = False # Freeze the base model layers to prevent re-training

model = Sequential([
    # Input shape: (Number of frames, Height, Width, Channels)
    Input(shape=(15, 128, 128, 3)),
    
    # Apply the CNN base to each frame in the video sequence individually
    TimeDistributed(base_cnn),
    
    # Condense the spatial features of each frame into a single vector
    TimeDistributed(GlobalAveragePooling2D()),
    
    # Use GRU to analyze the temporal patterns (changes over time) between frames
    GRU(64, return_sequences=False),
    
    # Regularization layer to prevent overfitting by randomly dropping 40% of neurons
    Dropout(0.4),
    
    # Fully connected layers for the final stress classification
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Binary output: Stressed (1) or Not Stressed (0)
])

# Configure the learning process: Using Adam optimizer and Binary Crossentropy for loss
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 2. Define Callbacks for Training Control
# Stop training early if the validation loss does not improve for 4 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=4, 
    restore_best_weights=True, # Roll back to the best version found during training
    verbose=1
)

# Save the best version of the model based on the highest Validation Accuracy
checkpoint = ModelCheckpoint(
    'stress_detection_best_model.keras', # Name of the file to save the model
    monitor='val_accuracy',              # Focus on tracking the validation accuracy
    save_best_only=True,                 # Only overwrite the file if the accuracy improves
    mode='max',                          # The goal is to maximize the monitored metric (accuracy)
    verbose=1                            # Print a message when a new best model is saved
)

# 3. Start the training process 
# the number of epochs to 30 because EarlyStopping will automatically 
# halt the process if the model stops improving, protecting it from overfitting.
history = model.fit(
    train_gen,            # The training data generator
    validation_data=test_gen, # The testing data generator for evaluation
    epochs=30,            # Maximum number of passes through the entire dataset
    callbacks=[early_stop, checkpoint] # Active monitors during training
)

I0000 00:00:1778518160.895070      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778518160.897860      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Load the optimal model saved during the training phase
print("Loading the best saved model...")
model = tf.keras.models.load_model('stress_detection_best_model.keras')

# 2. Evaluate model performance on the unseen test dataset
print("Evaluating model on test data...")
test_loss, test_acc = model.evaluate(test_gen)
print(f"\n Test Accuracy: {test_acc*100:.2f}%")
print(f" Test Loss: {test_loss:.4f}")

# 3. Generate predictions for performance analysis
print("Generating model predictions...")
# Predict probabilities and convert to binary labels (Threshold = 0.5)
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Extract ground truth labels from the generator for the processed batches
y_true = np.array(test_l[:len(y_pred)])

# 4. Visualize the Confusion Matrix
# This matrix shows the distribution of True Positives, True Negatives, etc.
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Stressed', 'Stressed'], 
            yticklabels=['Not Stressed', 'Stressed'])
plt.title('Confusion Matrix: Stress Detection')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.show()

# 5. Print a comprehensive Classification Report
# Includes Precision, Recall, and F1-Score for each category
print("\n Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Not Stressed', 'Stressed']))

# 6. Plot Learning History (Accuracy and Loss Curves)
# Essential for checking convergence and potential overfitting
plt.figure(figsize=(12, 4))

# Plot Accuracy over Epochs
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Training Accuracy History')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss over Epochs
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Training Loss History')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()